# Pythia 1b: Accessibility Domain Knowledge

16 layers | 8 heads | d_model=2048 | d_mlp=8192 | parallel attn+MLP

In [1]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [2]:
model_name = "pythia-1b"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-1b into HookedTransformer
Loaded pythia-1b on mps
  16 layers | 8 heads | d_model=2048 | d_mlp=8192 | parallel attn+MLP


In [3]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: pythia-1b
Prompt: "The W in WCAG stands for"
Target: " Web" (token 7066)
Final prediction: " Web"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0        king                 28367          0.000004    
1        ging                 46148          0.000000    
2         something           23446          0.000004    
3         something           33947          0.000001    
4         something           30718          0.000001    
5         “                   8410           0.000013    
6         abbrev              9856           0.000012    
7        ge                   17297          0.000006    
8         “                   9538           0.000013    
9         “                   1329           0.000094    
10       W                    79             0.000006    
11        W                   11             0.000936    
12        "                   14       

In [4]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [5]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/pythia/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/pythia/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/pythia/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/pythia/{model_name}/figures")

Saved: ../../results/pythia/pythia-1b/figures/wcag-logit-lens.png
Saved: ../../results/pythia/pythia-1b/figures/wcag-decomposition.png
Saved: ../../results/pythia/pythia-1b/figures/wcag-head-heatmap.png
Saved: ../../results/pythia/pythia-1b/figures/aria-logit-lens.png
Saved: ../../results/pythia/pythia-1b/figures/aria-decomposition.png
Saved: ../../results/pythia/pythia-1b/figures/aria-head-heatmap.png
Saved: ../../results/pythia/pythia-1b/figures/alt-logit-lens.png
Saved: ../../results/pythia/pythia-1b/figures/alt-decomposition.png
Saved: ../../results/pythia/pythia-1b/figures/alt-head-heatmap.png
Saved: ../../results/pythia/pythia-1b/figures/HTML-logit-lens.png
Saved: ../../results/pythia/pythia-1b/figures/HTML-decomposition.png
Saved: ../../results/pythia/pythia-1b/figures/HTML-head-heatmap.png
Saved: ../../results/pythia/pythia-1b/figures/screenReader-logit-lens.png
Saved: ../../results/pythia/pythia-1b/figures/screenReader-decomposition.png
Saved: ../../results/pythia/pythia-1b/fi

In [6]:
unload(model)

Model unloaded, memory cleared
